In [1]:
import time
def do_refocus(coord0_range, coord0_num, coord1_range, coord1_num, afm_scan_speed, counter_int_time):
    qafm_gui.disable_scan_actions_quanti()
    current_pos = afm_scanner_logic.get_afm_pos()
    coord0_origin = current_pos['x']
    coord1_origin = current_pos['y']
    xy_refocus_result = afm_scanner_logic.scan_true_area_AWG_qafm_fw_by_point(coord0_origin, coord0_range, coord0_num,
                                            coord1_origin, coord1_range, coord1_num, rotation = 0,
                                            afm_int_time=0, afm_scan_speed=afm_scan_speed, counter_int_time = counter_int_time,
                                            use_iso_B_mode = False, use_single_iso_B = False, iso_B_freq1 = 2.87e9, iso_B_freq2 = 2.87e9, mw_power = -20,
                                            liftoff_mode=False, liftoff_height=0,
                                            tip_osc_off = False, tip_osc_turn_off_time = 0, tip_osc_turn_on_time = 0)
    count_arr = xy_refocus_result['counts_fw']['data']
    x_start = xy_refocus_result['counts_fw']['coord0_arr'][0]
    x_stop = xy_refocus_result['counts_fw']['coord0_arr'][-1]
    y_start = xy_refocus_result['counts_fw']['coord1_arr'][0]
    y_stop = xy_refocus_result['counts_fw']['coord1_arr'][-1]
    x_max, y_max, c_max = afm_scanner_logic._calc_max_val_xy(arr= count_arr, 
                                                    x_start=x_start, x_stop=x_stop, 
                                                    y_start=y_start, y_stop=y_stop)
    afm_scanner_logic.set_afm_pos({'x': x_max, 'y': y_max})
    

In [2]:
check_period = 1800 #in second
coord0_range = 2e-6
coord0_num = 15
coord1_range = 2e-6
coord1_num = 15
afm_scan_speed = 40e-6 #in m s^-1
counter_int_time = 10e-3

## For pulse measurement AWG

In [44]:
refocus_start = time.time()
afm_scanner_logic.jupyter_meas_stop = False
while True:
    if afm_scanner_logic.jupyter_meas_stop:
        print('Sample refocus stopped!')
        break
    elif abs(refocus_start-time.time())>=check_period:
        pulsedmeasurementlogic_AWG.pause_pulsed_measurement()
        #do refocus here
        do_refocus(coord0_range, coord0_num, coord1_range, coord1_num, afm_scan_speed, counter_int_time)
        refocus_start = time.time()
        pulsedjupyterlogic_AWG.sample_load_ready_pulsestreamer(name='read_out_jptr')
        pulsedmeasurementlogic_AWG.continue_pulsed_measurement()
    
    else:
        time.sleep(0.001)

## For CW ODMR

In [11]:
refocus_start = time.time()
afm_scanner_logic.jupyter_meas_stop = False
while True:
    if afm_scanner_logic.jupyter_meas_stop:
        print('Sample refocus stopped!')
        break
    elif abs(refocus_start-time.time())>=check_period:
        odmrlogic.stop_odmr_scan()
        #do refocus here
        do_refocus(coord0_range, coord0_num, coord1_range, coord1_num, afm_scan_speed, counter_int_time)
        refocus_start = time.time()
        odmrlogic.continue_odmr_scan()
    else:
        time.sleep(0.001)

In [3]:
do_refocus(coord0_range, coord0_num, coord1_range, coord1_num, afm_scan_speed, counter_int_time)